# Multi-Provider Prompting Tutorial

## Overview

This tutorial demonstrates how to use multiple LLM providers with the prompt engineering techniques covered in this repository. By supporting different providers, you can compare model outputs, avoid vendor lock-in, and leverage each provider's unique strengths.

## Motivation

Most prompt engineering tutorials are tightly coupled to a single LLM provider (typically OpenAI). However, in practice, teams often evaluate prompts across multiple providers to:

- **Compare quality**: Different models may excel at different tasks.
- **Reduce costs**: Some providers offer competitive pricing for comparable quality.
- **Improve reliability**: Having a fallback provider ensures availability.
- **Explore options**: Newer providers like MiniMax offer high-performance models with OpenAI-compatible APIs.

## Key Components

1. **`utils/llm_provider.py`**: A helper module providing `get_llm()` that creates a LangChain-compatible LLM for any supported provider.
2. **Provider Auto-Detection**: Automatically selects the provider based on available API keys.
3. **Consistent Interface**: All providers return a `ChatOpenAI` instance, so existing prompt templates and chains work without modification.

## Supported Providers

| Provider | Models | API Key Env Var | Base URL |
|----------|--------|-----------------|----------|
| OpenAI | gpt-4o-mini, gpt-4o, etc. | `OPENAI_API_KEY` | (default) |
| MiniMax | MiniMax-M2.5, MiniMax-M2.7, MiniMax-M2.5-highspeed | `MINIMAX_API_KEY` | `https://api.minimax.io/v1` |

## Conclusion

By the end of this tutorial, you will:

1. Know how to configure and switch between LLM providers.
2. Be able to compare prompt outputs across providers.
3. Understand how to auto-detect providers from environment variables.
4. Have a reusable pattern for multi-provider prompt engineering.

## Setup

First, install the required dependencies and configure your API keys in a `.env` file.

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Add the project root to sys.path so we can import utils
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

load_dotenv()

from utils.llm_provider import get_llm, get_provider_name, SUPPORTED_PROVIDERS
from langchain.prompts import PromptTemplate

print(f"Supported providers: {', '.join(SUPPORTED_PROVIDERS)}")
print(f"Active provider: {get_provider_name()}")

## 1. Basic Provider Usage

The `get_llm()` function creates a LangChain `ChatOpenAI` instance for any supported provider. You can specify the provider explicitly, or let it auto-detect from the `LLM_PROVIDER` environment variable.

In [ ]:
# Auto-detect provider from environment
llm = get_llm()

# Or explicitly choose a provider:
# llm = get_llm(provider="openai")
# llm = get_llm(provider="minimax", model="MiniMax-M2.5")

prompt = PromptTemplate.from_template(
    "Explain the concept of {topic} in exactly two sentences."
)
chain = prompt | llm

result = chain.invoke({"topic": "prompt engineering"}).content
print(f"Provider: {get_provider_name()}")
print(f"Response: {result}")

## 2. Comparing Providers Side-by-Side

One of the most valuable applications of multi-provider support is comparing how different models respond to the same prompt. This helps you:
- Evaluate prompt robustness across models
- Identify provider-specific strengths
- Choose the best provider for your use case

In [ ]:
def compare_providers(prompt_template, variables, providers=None):
    """Run the same prompt across multiple providers and display results."""
    if providers is None:
        # Only compare providers whose API keys are available
        providers = []
        if os.getenv("OPENAI_API_KEY"):
            providers.append("openai")
        if os.getenv("MINIMAX_API_KEY"):
            providers.append("minimax")

    prompt = PromptTemplate.from_template(prompt_template)

    for provider in providers:
        try:
            llm = get_llm(provider=provider)
            chain = prompt | llm
            result = chain.invoke(variables).content
            print(f"=== {provider.upper()} ===")
            print(result)
            print()
        except ValueError as e:
            print(f"=== {provider.upper()} ===")
            print(f"Skipped: {e}")
            print()


# Compare a zero-shot classification prompt
compare_providers(
    prompt_template="""Classify the sentiment of the following text as Positive, Negative, or Neutral.
Respond with only the classification label.

Text: {text}

Sentiment:""",
    variables={"text": "The new park downtown is a wonderful addition to the neighborhood."},
)

## 3. Few-Shot Learning Across Providers

Few-shot prompts should work consistently across providers. Let's verify this with a classification task.

In [ ]:
few_shot_template = """Classify the following customer review into one of these categories:
Product Quality, Customer Service, Shipping, or Pricing.

Examples:
Review: "The fabric feels cheap and started fraying after one wash."
Category: Product Quality

Review: "The support team resolved my issue within 10 minutes."
Category: Customer Service

Review: "My order arrived a week late and the box was damaged."
Category: Shipping

Review: "At $15, this is a steal compared to similar products."
Category: Pricing

Now classify this review:
Review: "{review}"
Category:"""

compare_providers(
    prompt_template=few_shot_template,
    variables={"review": "I was charged twice for the same item and the refund took forever."},
)

## 4. Chain of Thought with Different Providers

Chain of Thought (CoT) prompting asks the model to reason step by step. Different providers may produce varying levels of reasoning detail.

In [ ]:
cot_template = """Solve the following problem step by step. Show your reasoning clearly.

Problem: {problem}

Solution:"""

compare_providers(
    prompt_template=cot_template,
    variables={
        "problem": (
            "A store offers a 20% discount on all items. "
            "If a jacket originally costs $85, and there is an additional "
            "8% sales tax applied after the discount, what is the final price?"
        )
    },
)

## 5. Role Prompting with Provider Selection

Role prompting assigns a persona to the model. Let's see how different providers interpret the same role.

In [ ]:
role_template = """You are a {role}. Based on your expertise, answer the following question.
Keep your response concise (3-4 sentences).

Question: {question}

Answer:"""

compare_providers(
    prompt_template=role_template,
    variables={
        "role": "senior cybersecurity analyst",
        "question": "What are the top three practices every organization should adopt to prevent phishing attacks?",
    },
)

## 6. Using MiniMax Directly

MiniMax offers high-performance models (M2.5 and M2.7) with an OpenAI-compatible API. Here is how to use MiniMax directly.

In [ ]:
# Create a MiniMax LLM instance
# Requires MINIMAX_API_KEY in your .env file
# Get your API key at https://platform.minimaxi.com/
try:
    minimax_llm = get_llm(provider="minimax", model="MiniMax-M2.5")

    prompt = PromptTemplate.from_template(
        """Generate a creative product name and a one-line tagline for the following concept.

Concept: {concept}

Product Name:
Tagline:"""
    )
    chain = prompt | minimax_llm

    result = chain.invoke(
        {"concept": "An AI-powered notebook that organizes your thoughts automatically"}
    ).content
    print("MiniMax Response:")
    print(result)

except ValueError as e:
    print(f"MiniMax not configured: {e}")
    print("Set MINIMAX_API_KEY in your .env file to use MiniMax.")
    print("Get your API key at https://platform.minimaxi.com/")

## 7. Switching Providers via Environment Variable

You can control the default provider by setting the `LLM_PROVIDER` environment variable. This is useful for CI/CD pipelines or team-wide configuration.

In [ ]:
# Demonstrate environment-based provider switching
original_provider = os.getenv("LLM_PROVIDER", "")

for provider_name in SUPPORTED_PROVIDERS:
    os.environ["LLM_PROVIDER"] = provider_name
    detected = get_provider_name()
    print(f"LLM_PROVIDER={provider_name!r} -> get_provider_name() = {detected!r}")

# Restore original
if original_provider:
    os.environ["LLM_PROVIDER"] = original_provider
else:
    os.environ.pop("LLM_PROVIDER", None)

print(f"\nRestored provider: {get_provider_name()!r}")

## Summary

In this tutorial, we explored:

1. **`get_llm()`** - A unified function to create LLM instances for any supported provider.
2. **Provider comparison** - Running the same prompt across providers to evaluate output quality.
3. **Technique portability** - Zero-shot, few-shot, CoT, and role prompting all work consistently across providers.
4. **Auto-detection** - The system automatically selects a provider based on available API keys.
5. **Environment configuration** - Use `LLM_PROVIDER` to control the default provider.

### Adding a New Provider

To add support for another OpenAI-compatible provider, simply add its configuration to `_PROVIDER_DEFAULTS` in `utils/llm_provider.py`. The same LangChain chains and prompts will work automatically.

### Recommended Models

| Provider | Model | Best For |
|----------|-------|----------|
| OpenAI | gpt-4o-mini | General use, cost-effective |
| OpenAI | gpt-4o | Complex reasoning, highest quality |
| MiniMax | MiniMax-M2.5 | General use, large 204K context |
| MiniMax | MiniMax-M2.7 | Latest model, best quality |
| MiniMax | MiniMax-M2.5-highspeed | Fast inference, 204K context |